# पाठ १६ - Microsoft Foundry सँग स्केलेबल एजेन्टहरू डिप्लोइ गर्न

यस नोटबुकमा तपाईंले काल्पनिक कम्पनी **Contoso** का लागि **उत्पादन-तयार ग्राहक समर्थन एजेन्ट** निर्माण गर्नुहुन्छ। पहिलेका पाठहरू जस्तो होइन, यहाँ एजेन्टको तर्क लूप होइन — यसको वरिपरि सबैकुरा छ जसले एजेन्टलाई स्केलमा सुरक्षित रूपमा चलाउन योग्य बनाउँछ:

१. **टूल कलिङ** — अर्डर खोजी र टिकट सिर्जना।
२. **RAG** — ज्ञान आधारबाट नीति उत्तरहरू।
३. **स्मृति** — ग्राहकलाई कुराकानी बीचमा सम्झिने।
४. **मोडेल मार्गदर्शन** — साधारण अनुरोधहरू साना मोडेलमा पठाउने, जटिलहरू ठूला मोडेलमा पठाउने।
५. **प्रतिक्रिया क्याचिंग** — मोडेल कल बिना बारम्बार प्रश्नहरूको सेवा गर्ने।
६. **मानव अनुमोदन** — थ्रेसहोल्ड माथिका फिर्ताहरूको लागि अनुमोदनको लागि रोक।
७. **मूल्यांकन गेट** — नराम्रो रिलिजलाई रोक्न अनलाइन टेस्ट सेट।
८. **अनुप्रयोग निरीक्षण** — प्रत्येक अनुरोध वरिपरि OpenTelemetry ट्रेसिङ।

प्रत्येक भाग स्व-सम्पूर्ण र चलाउन मिल्ने छ। प्रत्येक लाइन पढ्नुहोस् — उत्पादन प्रिमिटिभहरू जानाजानी साना राखिएका छन्।


## सेटअप

यो नोटबुक चलाउनुअघि, निश्चित गर्नुहोस् कि तपाईंले:

1. **एक Microsoft Foundry परियोजना** जसमा एक डिप्लोइड च्याट मोडेल छ (जस्तै `gpt-5-mini`)।
2. **Azure CLI बाट लगइन गर्नुभएको छ** — टर्मिनलमा `az login` चलाउनुहोस्।
3. **आवश्यक वातावरण भेरिएबलहरू सेट गर्नुभएको छ:**
   - `AZURE_AI_PROJECT_ENDPOINT` — तपाईंको Microsoft Foundry परियोजनाको अन्त्य बिन्दु।
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — तपाईंले डिप्लोइ गरेको मोडेलको नाम।

RAG खण्डले **Azure AI Search** प्रयोग गर्छ जब `AZURE_SEARCH_SERVICE_ENDPOINT` र `AZURE_SEARCH_API_KEY` सेट गरिएका छन्, र यदि छैन भने नोटबुकले इन-मेमोरी सर्च प्रयोग गर्छ जसले गर्दा खोज स्रोत बिना पनि नोटबुक चल्छ।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. उपकरणहरू

उत्पादन उपकरणहरूले वास्तविक प्रणालीहरूमा वास्तविक काम गर्छन्। यहाँ हामीले एक अर्डर डाटाबेस र एक टिकटिङ प्रणाली सादा Python फंक्शनहरू प्रयोग गरेर अनुकरण गरेका छौं। `@tool` डेकोरेटरले तिनीहरूलाई एजेन्टलाई उपलब्ध गराउँछ।

ध्यान दिनुहोस कि `issue_refund` ले `approval_mode="always_require"` प्रयोग गर्छ रिफन्डहरू थ्रेसहोल्डभन्दा माथि हुँदा — यो हो मानव-इन-द-लूप प्रिमिटिव जुन हामी पछि लागू गर्नेछौं।


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — नीति ज्ञान आधार

नीति सम्बन्धि प्रश्नहरू ("तपाईंको फिर्ता विन्डो कति हो?") अधिकारिक स्रोतबाट जवाफ दिइनु पर्छ, मोडेलको स्मृतिबाट होइन। हामी एक सानो ज्ञान आधारलाई खोज उपकरणको रूपमा ब्याक्छौ।

उत्पादनमा यो **Azure AI Search** हो; यहाँ हामी इन-मेमोरी कुञ्जीशब्द खोज प्रदान गर्दछौं ताकि नोटबुक जतै पनि चलोस्, र जब वातावरण भेरिएबलहरू उपलब्ध हुन्छन् तब स्वचालित रूपमा Azure AI Search मा सर्न्छ। 


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. स्मृति

एउटा सपोर्ट एजेन्ट जसले क्याम सँग कुरा गर्दैछ भनेर बिर्सिन्छ, त्यो खराब सपोर्ट एजेन्ट हो। हामी प्रत्येक ग्राहकका लागि सानो प्रोफाइल स्टोर राख्छौं र एजेन्टका निर्देशनहरूमा एउटा सानो सारांश इन्जेक्ट गर्छौं। उत्पादनमा यो एउटा मेमोरी सेवा हो (पाठ १३ हेर्नुहोस्); यहाँ एउटा dict ले ढाँचालाई देखाउँछ।


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## ४ र ५. मोडेल रुटिङ र प्रतिक्रिया क्यासिङ

एउटै अनुरोध ह्यान्डलरमा जोडिएका दुई लागत नियन्त्रणहरू:

- **रुटिङ**: सस्तो ह्युरिस्टिक वर्गीकरणले निर्णय गर्छ कि अनुरोधलाई सानो वा ठूलो मोडेल चाहिन्छ कि छैन।
- **क्यासिङ**: सामान्यीकृत पुनरावृत्ति प्रश्नहरू कुनै मोडेल कल बिना क्यासबाट सिधै सेवा गरिन्छ।

यहाँ वर्गीकरणकर्ता जानाजानी सरल राखिएको छ। उत्पादनमा तपाईं यसलाई ट्राफिक विरुद्ध प्रमाणित गर्नुहुनेछ र Foundry को Model Router सँग प्रतिस्थापन गर्न सक्नुहुन्छ।


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-5-nano
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-5-mini

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 र 8। एजेन्ट, मानव स्वीकृति, र अवलोकनयोग्यता

अब हामी माथिका उपकरणहरूबाट एजेन्टलाई एकत्रित गर्छौं र प्रत्येक अनुरोधलाई OpenTelemetry स्प्यानमा लपेट्छौं। `handle_support_request` कार्य उत्पादन अनुरोध ह्यान्डलर हो: क्यास → रूट → ट्रेस → रन → क्यास।

मानव स्वीकृति फ्रेमवर्कबाट ह्यान्डल गरिन्छ: किनभने `issue_refund` को `approval_mode="always_require"` छ, रन रोकियो र एउटा स्वीकृति अनुरोध देखाइन्छ जुन हामी स्पष्ट रूपमा समाधान गर्छौं।


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

# Build one agent per model tier so we can route by cost. The current agent-framework
# selects the model on the client, so each tier gets its own FoundryChatClient.
_TOOLS = [get_order_status, open_ticket, search_policies, issue_refund]
_agents_by_model: dict[str, object] = {}


def agent_for(model_name: str):
    if model_name not in _agents_by_model:
        client = FoundryChatClient(
            project_endpoint=endpoint,
            model=model_name,
            credential=AzureCliCredential(),
        )
        _agents_by_model[model_name] = client.as_agent(
            name="ContosoSupportAgent",
            instructions=SUPPORT_INSTRUCTIONS,
            tools=_TOOLS,
        )
    return _agents_by_model[model_name]


# Default agent (used by the evaluation gate, which does not route).
support_agent = agent_for(SMALL_MODEL)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await agent_for(chosen_model).run(prompt)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## ७. मूल्यांकन गेट

यो पाठबाट रिलिज गेट हो: एक अफलाइन परीक्षण सेटले एजेन्टलाई स्कोर गर्दछ, र पास दरले थ्रेसहोल्ड पार गरेमा मात्र तैनाती अघि बढ्छ। यहाँ स्कोरर एक साधारण कीवर्ड-ओभरल्याप जाँच हो जसले नोटबुकलाई आत्म-समावेशी राख्छ; उत्पादनमा तपाईंले एलएलएम-अस-जज वा फ्रेमवर्क मूल्यांकनकर्ता प्रयोग गर्नुहुनेछ (पाठ १० हेर्नुहोस्)।


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## सँगै राख्दैं: एक सिमुलेट गरिएको रिलिज

तलको सेलले सारा लूप देखाउँछ जुन पाठले वर्णन गर्छ: मूल्याङ्कन गेट चलाउनुहोस्, र केवल यो पास भएमा मात्र "डिप्लोय" गर्नुहोस्। यो नमुनालाई तपाईंले CI मा एउटा एजेन्ट संस्करणलाई Foundry Agent Service मा प्रवर्धन गर्नु अघि चलाउनुहुनेछ।


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## सारांश

तपाईंले सबै सञ्चालन चिन्ता समेटिएको उत्पादन-तयार ग्राहक समर्थन एजेन्ट तयार गर्नुभयो:

- **उपकरणहरू, RAG, र मेमोरी** एजेन्टलाई क्षमता र सन्दर्भ दिन्छ।
- **मोडेल राउटिङ र क्यासिङ** ले ढिलाइ र लागत नियन्त्रणमा राख्छ।
- **मानव स्वीकृति** ले ठूला रिफन्डजस्ता उच्च जोखिम कार्यहरूलाई सुरक्षा गर्छ।
- **मूल्यांकन गेट** ले खराब रिलिजहरू र Ship हुनु अघि रोक्छ।
- **ट्रेसिङ** ले हरेक अनुरोधलाई अवलोकनयोग्य बनाउँछ।

### चुनौती

यस एजेन्टलाई विस्तार गर्नुहोस्:

1. **धेरै मोडेलहरूलाई समर्थन गर्नुहोस्** — तेस्रो "सम्झना" तह थप्नुहोस् र तेस्मा उन्नती/गुनासोहरू राउट गर्नुहोस्।
2. **मूल्यांकन गेटहरू थप्नुहोस्** — `TEST_CASES` लाई रिफन्ड-स्वीकृति परिदृश्यहरू समावेश गर्न विस्तार गर्नुहोस् र गेटले regresion हरूलाई समात्छ भनि पुष्टि गर्नुहोस्।
3. **लागत-सजग राउटिङ थप्नुहोस्** — प्रत्येक अनुरोधको अनुमानित लागत (सानो, ठूलो, वा क्यास) ट्र्याक गर्नुहोस् र मिश्रित सोधपुछको ब्याचपछि लागत रिपोर्ट प्रिन्ट गर्नुहोस्।

अर्को पाठमा तपाईं विपरीत यात्रा गर्नुहुनेछ र Microsoft Foundry Local र Qwen सँग पूर्ण रूपमा आफ्नै मेसिनमा एजेन्ट चलाउनुहुनेछ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
